In [5]:
%pip install pandas numpy scikit-learn
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from pathlib import Path
import sys

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "utils"))
import verify_dataset
import bar_chart
import train_models
import importlib
import sys
df = pd.read_csv(
 "data/payment_dataset.csv"
)
# ==========================================
# STEP 1: LOAD THE DATASET
# ==========================================
print(df.head())
print(df.shape)
print(df.isnull().sum().sum())

# =============================================================
### Step 2 - Verifying the dataset is there any missing values 
# =============================================================
print("\nMissing values:")
print(df.isnull().sum())
### calculating the count of "isFraud"
print("\nFraud distribution:")
print(df["isFraud"].value_counts())
### calculating the percentage of "isFraud"

print("\nFraud percentage:")
print(df["isFraud"].mean() * 100)

# =============================================================
### STEP 3: Data Engineering  EDP
# =============================================================
importlib.reload(bar_chart)
bar_chart.fraud_class_counts(
    df, filename="fraud_class_counts.png"
)
bar_chart.transaction_amount_distribution(
    df, filename="amount_distribution.png"
)

bar_chart.amount_by_fraud_status(
    df, filename="amount_by_fraud.png"
)

bar_chart.fraud_rate_by_type(
    df, filename="fraud_rate_by_type.png"
)

bar_chart.transaction_type_counts(
    df, filename="transaction_type_counts.png"
)

bar_chart.numeric_correlation_heatmap(
    df, filename="correlation_heatmap.png"
)

# ==========================================
# STEP 4: SELECT FEATURES AND TARGET
# ==========================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

df = pd.read_csv(
 "data/payment_dataset.csv"
)
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

X = df[features]
y = df["isFraud"]

# ==========================================
# STEP 5: SPLIT THE DATA
# 70% training, 15% validation, 15% testing
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = ["type"]
numeric_features = [c for c in features if c not in categorical_features]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

results = []
# ==========================================
# STEP 6: PREPROCESS THE FEATURES
# ==========================================
for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "PR-AUC (Average Precision)": average_precision_score(
            y_test, probabilities
        )
    }

    results.append(row)

    print(f"\n{name}")
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

results_df = pd.DataFrame(results)
display(results_df.round(4))
print("results_df",results_df)

# ==========================================
# STEP 7: Calculate the confusion matrix
# ==========================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
# Use the fitted Logistic Regression pipeline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Genuine", "Fraud"]
).plot()

plt.show()

# ===============================================
# STEP 8:Train the neural network - Deep learning
# ===============================================
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler, OneHotEncoder
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

categorical_features = ["type"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor created successfully!")

tf.keras.utils.set_random_seed(42)


# ===============================================
# STEP 9: Fit preprocessing on training data only
# ===============================================
nn_preprocessor = preprocessor

X_train_nn = nn_preprocessor.fit_transform(X_train)
X_test_nn = nn_preprocessor.transform(X_test)

# Convert sparse matrices to dense arrays for this small network.
# Check memory availability before using this on the full dataset.
if hasattr(X_train_nn, "toarray"):
    X_train_nn = X_train_nn.toarray()
    X_test_nn = X_test_nn.toarray()

X_train_nn = X_train_nn.astype("float32")
X_test_nn = X_test_nn.astype("float32")

y_train_nn = y_train.to_numpy().astype("float32")
y_test_nn = y_test.to_numpy().astype("float32")

classes = np.unique(y_train_nn)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_nn
)

class_weights = {
    int(label): float(weight)
    for label, weight in zip(classes, weights)
}

nn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_nn.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

history = nn_model.fit(
    X_train_nn,
    y_train_nn,
    epochs=5,
    batch_size=2048,
    validation_split=0.1,
    class_weight=class_weights,
    verbose=1
)
# ===============================================
# STEP 10: Evaluate the neural network
# ===============================================
nn_probabilities = nn_model.predict(
    X_test_nn,
    batch_size=2048
).ravel()

nn_predictions = (nn_probabilities >= 0.5).astype(int)

print(classification_report(
    y_test_nn,
    nn_predictions,
    zero_division=0
))

print("PR-AUC:", average_precision_score(
    y_test_nn, nn_probabilities
))

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

results = []

for threshold in thresholds:

    predictions = (
        nn_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_nn,
        predictions,
        labels=[0, 1]
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test_nn, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test_nn, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test_nn, predictions, zero_division=0
        ),
        "False Alarms": fp,
        "Missed Fraud": fn,
        "Fraud Detected": tp
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))
